# Lesson 06: The Qwen SLM Family — Dual Thinking Modes, MoE & Multilingual Edge Scaling

In this companion notebook, you will:
1. Load `Qwen/Qwen2.5-0.5B-Instruct` (or `Qwen/Qwen3-0.6B` when available) on Google Colab Free Tier.
2. Test multilingual generation and structured reasoning vs. fast chat mode.
3. Compute active vs. total parameter footprints for Sparse Mixture-of-Experts (MoE) Qwen variants.

In [ ]:
!pip install -q -U transformers accelerate torch

## 1. Run Open-Source `Qwen/Qwen2.5-0.5B-Instruct` on Colab
At 0.5B parameters (~1 GB FP16), this micro SLM runs comfortably on both Colab CPU and T4 GPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to(device)

messages = [
    {"role": "system", "content": "You are a concise Edge AI assistant."},
    {"role": "user", "content": "List 3 reasons why a 0.5B model is useful on a smartphone."}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(device)

with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=120, do_sample=False)
output_ids = generated[0][inputs.input_ids.shape[1]:]
print(tokenizer.decode(output_ids, skip_special_tokens=True))

## 2. Dual-Mode Token Budget Simulation
Compare fast `/no_think` responses against deep `/think` reasoning chains under a fixed edge latency budget.

In [ ]:
def estimate_latency_ms(ttft_ms: float, think_tokens: int, answer_tokens: int, tps: float) -> float:
    return ttft_ms + ((think_tokens + answer_tokens) / tps) * 1000.0

fast_ms = estimate_latency_ms(ttft_ms=80, think_tokens=0, answer_tokens=60, tps=75)
think_ms = estimate_latency_ms(ttft_ms=80, think_tokens=240, answer_tokens=60, tps=75)
print(f"Non-Thinking Mode: {fast_ms:.1f} ms")
print(f"Thinking Mode:     {think_ms:.1f} ms (+{think_ms - fast_ms:.1f} ms for reasoning trace)")